In [2]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns

# Import các thư viện mô hình
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from catboost import CatBoostClassifier

# Import thư viện đánh giá
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, roc_curve, accuracy_score
from sklearn.model_selection import PredefinedSplit, GridSearchCV

In [3]:

train_csv = pd.read_csv('../data/processed/train.csv')
valid_csv = pd.read_csv('../data/processed/valid.csv') # <--- MỚI
test_csv = pd.read_csv('../data/processed/test.csv')

print(f"Shape Train: {train_csv.shape}")
print(f"Shape Valid: {valid_csv.shape}")
print(f"Shape Test:  {test_csv.shape}")

# 2. TIỀN XỬ LÝ
target_col = "SOURCE"

# Hàm xử lý chung để đảm bảo nhất quán
def preprocess_data(df, is_train=False, train_columns=None):
    # Tách X, y
    X = df.drop(columns=[target_col])
    y = df[target_col].map({"in": 0, "out": 1}).astype(int)
    
    # One-Hot Encoding
    X = pd.get_dummies(X, drop_first=True)
    
    # Nếu không phải tập train, phải căn chỉnh cột theo tập train
    if not is_train:
        X = X.reindex(columns=train_columns, fill_value=0)
        
    return X, y

# Xử lý Train trước để lấy danh sách cột chuẩn
X_train, y_train = preprocess_data(train_csv, is_train=True)
train_cols = X_train.columns

# Xử lý Valid và Test theo cột của Train
X_valid, y_valid = preprocess_data(valid_csv, is_train=False, train_columns=train_cols)
X_test, y_test = preprocess_data(test_csv, is_train=False, train_columns=train_cols)

print("-" * 30)
print(f"X_train: {X_train.shape}")
print(f"X_valid: {X_valid.shape}")
print(f"X_test:  {X_test.shape}")

Shape Train: (3750, 11)
Shape Valid: (662, 11)
Shape Test:  (662, 11)
------------------------------
X_train: (3750, 10)
X_valid: (662, 10)
X_test:  (662, 10)


In [4]:
# 1. Gộp Train và Valid lại
X_combined = pd.concat([X_train, X_valid], axis=0)
y_combined = pd.concat([y_train, y_valid], axis=0)

# 2. Tạo chỉ mục (index) để báo cho GridSearchCV biết đâu là Train, đâu là Valid
# Giá trị -1: Dùng để Train
# Giá trị 0:  Dùng để Validate
# (Lưu ý: GridSearchCV mặc định sẽ train lại trên toàn bộ X_combined sau khi tìm ra tham số tốt nhất)
split_index = [-1] * len(X_train) + [0] * len(X_valid)
pds = PredefinedSplit(test_fold=split_index)

print(f"Tổng số mẫu combined: {len(X_combined)}")
print(f"Kích thước split_index: {len(split_index)}")

Tổng số mẫu combined: 4412
Kích thước split_index: 4412


In [5]:
# Định nghĩa các tham số cần tinh chỉnh cho từng mô hình
param_grids = {
    "Logistic Regression": {
        "model": LogisticRegression(random_state=42, max_iter=2000),
        "params": {
            "C": [0.01, 0.1, 1, 10],  # Độ mạnh của Regularization
            "solver": ['liblinear', 'lbfgs']
        }
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            "max_depth": [3, 5, 7, 10, None],
            "min_samples_split": [2, 5, 10],
            "criterion": ["gini", "entropy"]
        }
    },
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators": [50, 100, 200], # Số lượng cây
            "max_depth": [5, 10, 15, None],
            "min_samples_leaf": [1, 2, 4]
        }
    },
    "XGBoost": {
        "model": xgb.XGBClassifier(random_state=42, eval_metric='logloss'),
        "params": {
            "n_estimators": [50, 100, 200],
            "learning_rate": [0.01, 0.05, 0.1, 0.2],
            "max_depth": [3, 5, 7]
        }
    },
    # CatBoost chạy grid search khá lâu, ta thử ít tham số thôi
    "CatBoost": {
        "model": CatBoostClassifier(random_seed=42, verbose=0),
        "params": {
            "iterations": [100, 200],
            "learning_rate": [0.05, 0.1],
            "depth": [4, 6, 8]
        }
    }
}

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
import pandas as pd

final_results = []

print(f"{'Model':<20} | {'Best Params':<40} | {'Test Accuracy':<10}")
print("-" * 85)

for name, config in param_grids.items():

    # GridSearch chỉ đánh giá trên VALIDATION do PredefinedSplit
    clf = GridSearchCV(
        estimator=config["model"],
        param_grid=config["params"],
        cv=pds,                  # PredefinedSplit: Valid cố định
        scoring='accuracy',
        n_jobs=-1,
        refit=False             # Không tự động refit để tránh train lại với param cũ
    )

    # Bước 1: Tìm tham số tốt nhất dựa trên Train → Valid
    clf.fit(X_combined, y_combined)

    best_params = clf.best_params_
    best_score_valid = clf.best_score_

    # Bước 2: TỰ TRAIN LẠI từ đầu bằng tham số tốt nhất trên (Train + Valid)
    best_model = config["model"].set_params(**best_params)
    best_model.fit(X_combined, y_combined)

    # Bước 3: Dự đoán trên Test (100% unseen)
    y_test_pred = best_model.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)

    final_results.append({
        "Model": name,
        "Best Params": str(best_params),
        "Valid Accuracy": best_score_valid,
        "Test Accuracy": test_acc
    })

    print(f"{name:<20} | {str(best_params):<40} | {test_acc:.4f}")

# Hiển thị final table
print("\n--- KẾT QUẢ CHI TIẾT SAU KHI TUNING ---")
df_final = pd.DataFrame(final_results).sort_values(by="Test Accuracy", ascending=False)
print(df_final)


Model                | Best Params                              | Test Accuracy
-------------------------------------------------------------------------------------
Logistic Regression  | {'C': 0.01, 'solver': 'liblinear'}       | 0.7341
Decision Tree        | {'criterion': 'gini', 'max_depth': 7, 'min_samples_split': 2} | 0.7991
Random Forest        | {'max_depth': 15, 'min_samples_leaf': 4, 'n_estimators': 100} | 0.9184
XGBoost              | {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 100} | 0.9018
CatBoost             | {'depth': 6, 'iterations': 200, 'learning_rate': 0.1} | 0.8701

--- KẾT QUẢ CHI TIẾT SAU KHI TUNING ---
                 Model                                        Best Params  \
2        Random Forest  {'max_depth': 15, 'min_samples_leaf': 4, 'n_es...   
3              XGBoost  {'learning_rate': 0.2, 'max_depth': 5, 'n_esti...   
4             CatBoost  {'depth': 6, 'iterations': 200, 'learning_rate...   
1        Decision Tree  {'criterion': 'gini', 